Import Packages

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import glob
import os
from sklearn.model_selection import train_test_split

Create a Pandas and Polars object for each file in data

In [6]:
file_dfs = {}
file_dfs_pl = {}

names = [
         '../data/datascope_price_eod.parquet',
         '../data/fmp_hist_financial_statements.parquet',
         '../data/yieldbook_price_yield_with_labels.parquet'
        ]
# for name in glob.glob("../data/*.parquet"):
for name in names:
    filename = os.path.basename(name)
    filename = filename.removesuffix(".parquet")
    # file_dfs[filename] = pd.read_parquet(name).copy()
    file_dfs_pl[filename] = pl.read_parquet(name).clone()

Simple spot to view data

In [3]:
file_dfs_pl['datascope_yield_eod']

RIC,Trade Date,Open Yield,Low Yield,High Yield,Ask Yield,Bid Yield,Close Yield
str,datetime[ns],f64,f64,f64,f64,f64,f64
"""001055AY8=""",2018-10-30 00:00:00,null,null,null,null,null,null
"""001055AY8=""",2018-10-31 00:00:00,4.911,4.892,4.914,4.856,4.906,4.906
"""001055AY8=""",2018-11-01 00:00:00,4.916,4.849,4.936,4.826,4.876,4.876
"""001055AY8=""",2018-11-02 00:00:00,4.887,4.884,4.943,4.8711,4.9211,4.921
"""001055AY8=""",2018-11-05 00:00:00,4.909,4.882,4.914,4.84,4.89,4.89
…,…,…,…,…,…,…,…
"""US046686063=""",2025-09-24 00:00:00,5.017,5.006,5.092,4.986,5.016,5.016
"""US046686063=""",2025-09-25 00:00:00,4.994,4.986,5.14,5.0997,5.1297,5.13
"""US046686063=""",2025-09-26 00:00:00,5.105,5.081,5.122,5.0808,5.1108,5.111


Labels of each dataframe from the df dict

In [4]:
file_dfs.keys()

dict_keys(['datascope_characteristic_eod', 'datascope_eod_resampled_from_tick_complete', 'datascope_eod_us_treasury', 'datascope_metadata_eod_resampled_from_tick', 'datascope_price_eod', 'datascope_price_eod_resampled_from_tick', 'datascope_risk_eod_resampled_from_tick', 'datascope_yield_eod', 'datascope_yield_eod_resampled_from_tick', 'fmp_corp_actions', 'fmp_enterprise_valuations_ratings', 'fmp_hist_financial_statements', 'fmp_market_performance', 'issuer_reference', 'norgate_eod_stock_prices', 'wrds_eod_us_treasury_daily', 'wrds_eod_us_treasury_monthly', 'yieldbook_indicatives', 'yieldbook_price_yield'])

Similarly, test for viewing data

In [5]:
x = file_dfs["datascope_price_eod"]
x
# for column in x.columns:
#     print(column)

,RIC,Trade Date,Open Ask,Open Bid,Ask High,Bid High,Low Bid,Ask,Bid,Average of Bid and Ask,Universal Ask Price,Universal Bid Price,Universal Close Price
0,001055AY8=,2018-10-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,001055AY8=,2018-10-31,98.2580,97.4860,98.5540,97.7780,97.4400,98.3230,97.5490,97.9360,98.3230,97.5490,97.5490
2,001055AY8=,2018-11-01,98.1670,97.3950,99.2150,98.4310,97.0890,98.7910,98.0120,98.4015,98.7910,98.0120,98.0120
3,001055AY8=,2018-11-02,98.6190,97.8420,98.6660,97.8880,96.9820,98.0870,97.3170,97.7020,98.0870,97.3170,97.3170
4,001055AY8=,2018-11-05,98.2740,97.5010,98.7100,97.9320,97.4390,98.5850,97.8090,98.1970,98.5850,97.8090,97.8090
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2193456,US046686063=,2025-09-24,105.1994,104.8857,105.3135,104.9994,104.0975,105.2050,104.8913,105.0482,105.2050,104.8913,104.8913
2193457,US046686063=,2025-09-25,105.4410,105.1264,105.5134,105.1985,103.6018,104.0217,103.7125,103.8671,104.0217,103.7125,103.7125
2193458,US046686063=,2025-09-26,104.2775,103.9675,104.5262,104.2152,103.7899,104.2152,103.9054,104.0603,104.2152,103.9054,103.9054
2193459,US046686063=,2025-09-29,104.2630,103.9531,104.9663,104.6537,103.9531,104.9525,104.6399,104.7962,104.9525,104.6399,104.6399


In order to populate much of the daily data I needed to pull from the existing monthly data tables. Joining these on "RIC" and "Trade Date" as I am treating RIC as the primary id.
Should I find more columns of use later on, I could add them onto the existing merge 

Since "datascope_yield_eod" is lacking some of the data, I look for monthly matches that I can add. 

In [6]:
daily_df = file_dfs["datascope_yield_eod"][["RIC", "Trade Date", "Close Yield"]]
monthly_df = file_dfs["yieldbook_price_yield"][["RIC", "PricingDate","CurrentCoupon", "YearsToMaturity"]] #, "EffectiveDuration"
monthly_df = monthly_df.rename(columns={'PricingDate':"Trade Date"}) # Requires assumption that these dates are synchronous
monthly_df["Trade Date"] = monthly_df["Trade Date"].astype('datetime64[ns]') # Forces same data type
daily_df = pd.merge(daily_df, monthly_df, how='left', left_on=["RIC", "Trade Date"], right_on=["RIC", "Trade Date"])
daily_2_df = file_dfs["datascope_price_eod"][["RIC", "Trade Date", "Universal Close Price"]] # Identified other daily data that would be relevant for this merge. So "daily_2" is datascope_price_eod
daily_df = pd.merge(daily_df, daily_2_df, how='left', left_on=["RIC", "Trade Date"], right_on=["RIC", "Trade Date"])
daily_df["CurrentCoupon"] = daily_df["CurrentCoupon"].bfill() # Much of the data in these columns needs to be refined. For current coupon and years to maturity, the monthly data represents a cover for each days in that period. Precision
daily_df["YearsToMaturity"] = daily_df["YearsToMaturity"].bfill()
daily_df["Close Yield"] = daily_df["Close Yield"].ffill()
daily_df["Universal Close Price"] = daily_df["Universal Close Price"].ffill()
daily_df = daily_df.iloc[1:] #The first day had empty values. Given size of dataset, dropping seemed appropriate

PV- = [Coupon/(1 + Yield - delta curve)^1] + ... [Coupon + Par Value/ (1 + Yield - delta Curve)^Maturity]

Daily effective duration is certainly an important metric to estimate. The challenge is owed to the mismatch in periodicity of data. The formula belows attempts to calculate each for each day, moving row by row through the above daily_df. 

There were some computational simplifications made, such as choosing i within a years range. 

In [8]:
def effective_duration(df):
    # PV- = [Coupon/(1 + Yield - delta curve)^1] + ... [Coupon + Par Value/ (1 + Yield - delta Curve)^Maturity]
    close_yield = df["Close Yield"]/100
    coupon = df["CurrentCoupon"]/100
    years = int(df["YearsToMaturity"])
    delta = 0.01
    PV_plus = 0
    PV_zero = df["Universal Close Price"]
    PV_minus = 0
    for i in range(1, years+1):
        if i != years:
            PV_plus += (coupon/(1 + close_yield + delta)**i)
            PV_minus += (coupon/(1 + close_yield - delta)**i)
        else:
            PV_plus += (coupon+1)/((1 + close_yield + delta)**i)
            PV_minus += (coupon+1)/((1 + close_yield - delta)**i)
    eff_dur = (PV_minus-PV_plus)/(2*delta*PV_zero)
    return eff_dur*100

Create Effective Duration column within my daily df using the above function

In [9]:
daily_df["Effective Duration"] = daily_df.apply(effective_duration, axis=1)
daily_df

,RIC,Trade Date,Close Yield,CurrentCoupon,YearsToMaturity,Universal Close Price,Effective Duration
1,001055AY8=,2018-10-31,4.906,4.75,30.1945,97.5490,15.805380
2,001055AY8=,2018-11-01,4.876,4.75,30.1096,98.0120,15.840122
3,001055AY8=,2018-11-02,4.921,4.75,30.1096,97.3170,15.788298
4,001055AY8=,2018-11-05,4.890,4.75,30.1096,97.8090,15.821726
5,001055AY8=,2018-11-06,4.846,4.75,30.1096,98.4920,15.872676
...,...,...,...,...,...,...,...
2193623,US046686063=,2025-09-24,5.016,5.50,14.2877,104.8913,9.748470
2193624,US046686063=,2025-09-25,5.130,5.50,14.2877,103.7125,9.717756
2193625,US046686063=,2025-09-26,5.111,5.50,14.2877,103.9054,9.723097
2193626,US046686063=,2025-09-29,5.040,5.50,14.2877,104.6399,9.742171


Similar to the above goals, the below tables use the "fmp_hist_financial_statements". 

In [10]:
shortened_fmp_hist_financial_statements = file_dfs["fmp_hist_financial_statements"][["RIC", "date", "priceToBookRatio", "priceToEarningsRatio", "totalDebt", "totalEquity", "reportedCurrency"]].sort_values(by=["RIC", "date"])
shortened_fmp_hist_financial_statements = shortened_fmp_hist_financial_statements[shortened_fmp_hist_financial_statements.reportedCurrency == "USD"]
shortened_fmp_hist_financial_statements = shortened_fmp_hist_financial_statements.dropna(subset=["priceToBookRatio","priceToEarningsRatio"])
shortened_fmp_hist_financial_statements = shortened_fmp_hist_financial_statements.rename(columns={"date":"Trade Date","priceToBookRatio":"P/B", "priceToEarningsRatio":"P/E"})
shortened_fmp_hist_financial_statements["Market Leverage"] = (shortened_fmp_hist_financial_statements["totalDebt"])/(shortened_fmp_hist_financial_statements["totalEquity"])
shortened_fmp_hist_financial_statements = shortened_fmp_hist_financial_statements.drop(columns=["totalDebt", "totalEquity", "reportedCurrency"], axis=1)
shortened_fmp_hist_financial_statements

,RIC,Trade Date,P/B,P/E,Market Leverage
1544,001055BJ0=,2017-03-31,1.428212,12.267667,0.258112
1545,001055BJ0=,2017-06-30,1.432122,10.797656,0.244245
1546,001055BJ0=,2017-09-30,1.461100,11.211798,0.238795
1547,001055BJ0=,2017-12-31,1.412835,3.331757,0.217074
1549,001055BJ0=,2018-03-31,1.402781,11.879131,0.221188
...,...,...,...,...,...
18332,US046294726=,2024-03-31,4.824640,12.245253,0.915678
18333,US046294726=,2024-06-30,4.334703,15.012559,0.882421
18334,US046294726=,2024-09-30,3.857862,20.956790,0.819686
18335,US046294726=,2024-12-31,3.652018,18.893466,0.788136


Further expand daily df to contain more of the relevant characteristics. This time combining with financial statement data like P/B, P/E, and Leverage

In [11]:
daily_df = pd.merge(daily_df, shortened_fmp_hist_financial_statements, how='left', left_on=["RIC", "Trade Date"], right_on=["RIC", "Trade Date"])
daily_df


,RIC,Trade Date,Close Yield,CurrentCoupon,YearsToMaturity,Universal Close Price,Effective Duration,P/B,P/E,Market Leverage
0,001055AY8=,2018-10-31,4.906,4.75,30.1945,97.5490,15.805380,NaN,NaN,NaN
1,001055AY8=,2018-11-01,4.876,4.75,30.1096,98.0120,15.840122,NaN,NaN,NaN
2,001055AY8=,2018-11-02,4.921,4.75,30.1096,97.3170,15.788298,NaN,NaN,NaN
3,001055AY8=,2018-11-05,4.890,4.75,30.1096,97.8090,15.821726,NaN,NaN,NaN
4,001055AY8=,2018-11-06,4.846,4.75,30.1096,98.4920,15.872676,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2193622,US046686063=,2025-09-24,5.016,5.50,14.2877,104.8913,9.748470,NaN,NaN,NaN
2193623,US046686063=,2025-09-25,5.130,5.50,14.2877,103.7125,9.717756,NaN,NaN,NaN
2193624,US046686063=,2025-09-26,5.111,5.50,14.2877,103.9054,9.723097,NaN,NaN,NaN
2193625,US046686063=,2025-09-29,5.040,5.50,14.2877,104.6399,9.742171,NaN,NaN,NaN


Since financial statement data will be referenced until the next print, I fill the data based on these data releases

In [12]:
RIC_df = daily_df["RIC"]
daily_df = daily_df.groupby("RIC").ffill()
daily_df["RIC"] = RIC_df
cols = list(daily_df)
cols.insert(0, cols.pop(cols.index('RIC')))
daily_df = daily_df.loc[:, cols]
daily_df

,RIC,Trade Date,Close Yield,CurrentCoupon,YearsToMaturity,Universal Close Price,Effective Duration,P/B,P/E,Market Leverage
0,001055AY8=,2018-10-31,4.906,4.75,30.1945,97.5490,15.805380,NaN,NaN,NaN
1,001055AY8=,2018-11-01,4.876,4.75,30.1096,98.0120,15.840122,NaN,NaN,NaN
2,001055AY8=,2018-11-02,4.921,4.75,30.1096,97.3170,15.788298,NaN,NaN,NaN
3,001055AY8=,2018-11-05,4.890,4.75,30.1096,97.8090,15.821726,NaN,NaN,NaN
4,001055AY8=,2018-11-06,4.846,4.75,30.1096,98.4920,15.872676,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2193622,US046686063=,2025-09-24,5.016,5.50,14.2877,104.8913,9.748470,NaN,NaN,NaN
2193623,US046686063=,2025-09-25,5.130,5.50,14.2877,103.7125,9.717756,NaN,NaN,NaN
2193624,US046686063=,2025-09-26,5.111,5.50,14.2877,103.9054,9.723097,NaN,NaN,NaN
2193625,US046686063=,2025-09-29,5.040,5.50,14.2877,104.6399,9.742171,NaN,NaN,NaN


In [13]:
### Copied labels_preparation workflow for testing
yield_book = pd.read_parquet(r'../data/yieldbook_price_yield.parquet')
yield_book = yield_book.dropna(subset=['PricingDate'])
yield_book['PricingDate'] = pd.to_datetime(yield_book['PricingDate'], format='%Y-%m-%d')
yield_book['MONTH'] = yield_book['PricingDate'].dt.to_period("M")
treasury_bond_crsp = pd.read_parquet(r"../data/wrds_eod_us_treasury_monthly.parquet")
treasury_data = treasury_bond_crsp[['TCUSIP','MCALDT','TDATDT','TMDURATN','TCOUPRT','TNIPPY','TMYLD','TMNOMPRC','TMRETNUA']].dropna(axis=0)
# Converting Daily Macualay Duration to Annual Modified Duration
treasury_data = treasury_data.dropna(subset=['TMRETNUA'])
treasury_data['TMDURATN'] = treasury_data['TMDURATN'] / 365
treasury_data['DURATION'] = np.where(treasury_data['TNIPPY'] != 0, treasury_data['TMDURATN'] / (1+(treasury_data['TMYLD']/treasury_data['TNIPPY'])), treasury_data['TMDURATN'])
treasury_data['MCALDT'] = pd.to_datetime(treasury_data['MCALDT'], format='%Y-%m-%d')
treasury_data['MONTH'] = treasury_data['MCALDT'].dt.to_period("M")
# Bucketing treasury bonds in buckets of 3 months based on their modified duration to find on the run bond in that bucket
def round_to_nearest_quarter(value):
    """
    Rounds a given number to the nearest quarter (0.25).
    """
    return round(value / 0.25) * 0.25
treasury_data['DURATION_BUCKET'] = treasury_data['DURATION'].apply(round_to_nearest_quarter)
on_the_run = treasury_data.groupby(['MONTH','DURATION_BUCKET'])['TDATDT'].max().reset_index()
treasury_data_otr = pd.merge(treasury_data, on_the_run, on=['MONTH','DURATION_BUCKET','TDATDT'], how='right')
# Checking if there are more than 1 on the run bond in each bucket
counts = treasury_data_otr.groupby(['MONTH','DURATION_BUCKET'])['TCUSIP'].count()
treasury_data_otr = treasury_data_otr.sort_values(['MONTH','DURATION'])
treasury_data_otr = treasury_data_otr.set_index('MONTH')
# We calculate the weights based on a duration-based hedge using treasury bonds of nearest duration (linear interpolation)
def calc_weights(row):
    curr_month = row['MONTH']
    curr_month_data = treasury_data_otr.loc[curr_month]

    next_month = curr_month + 1
    try:
        next_month_cusips = treasury_data_otr.loc[next_month, 'TCUSIP'].tolist()
    except:
        return {}

    curr_month_data = curr_month_data.loc[curr_month_data['TCUSIP'].isin(next_month_cusips)]

    x = curr_month_data['DURATION']
    y = curr_month_data['TCUSIP']

    lower_ind = x.loc[x<row['Duration']].shape[0] - 1
    if lower_ind == x.shape[0] - 1:
        return {y.iloc[lower_ind]: 1}
    upper_ind = lower_ind + 1

    w1 = (x.iloc[upper_ind] - row['Duration']) / (x.iloc[upper_ind] - x.iloc[lower_ind])
    w2 = 1 - w1

    return {y.iloc[lower_ind]: w1, y.iloc[upper_ind]: w2}
# Calculating the hedge weights at T timepoint
yield_book['Hedge_Weights(T)'] = yield_book.apply(calc_weights, axis=1)
yield_book = yield_book.sort_values(by=['CUSIP','MONTH'])

# Calculate the bond returns based on price change
# To-do: Get TotalReturn column from LSEG
yield_book['Returns'] = yield_book.groupby(['CUSIP'])['FullPrice'].pct_change()

# Calculating Duration x Spread using SpreadDuration and OAS based on T-1 timepoint
yield_book['DtS'] = yield_book.groupby(['CUSIP'])[['SpreadDuration','OAS']].shift(1).prod(axis=1)

# Shifting the hedge weights to calculate hedge returns based on T-1 timepoint
yield_book['Hedge_Weights(T-1)'] = yield_book.groupby(['CUSIP'])['Hedge_Weights(T)'].shift(1)
def calc_hedge_return(row):
    weights = row['Hedge_Weights(T-1)']
    if not isinstance(weights, dict):
        return np.nan
    curr_month = row['MONTH']

    hedge_return = 0
    for cusip, weight in weights.items():
        ret = treasury_data_otr.loc[curr_month]
        ret = ret.loc[(ret['TCUSIP']==cusip), 'TMRETNUA'].iloc[0]
        if ret == '' or np.isnan(ret):
            return float('nan')
        hedge_return += ret * weight

    return hedge_return

# Calculating hedge returns based on T-1 timepoint duration hedge weights
yield_book['Hedge_Returns'] = yield_book.apply(calc_hedge_return, axis=1)
# Calculating raw credit excess return based on bond returns and hedge returns (based on T-1 timepoint duration hedge weights)
yield_book['Raw_Credit_Excess_Return'] = yield_book['Returns'] - yield_book['Hedge_Returns']

yield_book = yield_book.dropna(subset=['Raw_Credit_Excess_Return', 'DtS'])

# Scaling raw credit excess return based on Duration x Spread to reduce the effect of volatile bonds in final output
yield_book['Credit_Excess_Return'] = yield_book['Raw_Credit_Excess_Return'] / yield_book['DtS'].apply(lambda x: max(0.25, x))

In [14]:
daily_df.sort_values(by="Trade Date")

,RIC,Trade Date,Close Yield,CurrentCoupon,YearsToMaturity,Universal Close Price,Effective Duration,P/B,P/E,Market Leverage
2169448,US013039666=,2002-01-31,4.130,7.45,13.1507,106.1100,11.280673,NaN,NaN,NaN
2169449,US013039666=,2002-02-01,4.130,7.45,13.1507,106.3700,11.253100,NaN,NaN,NaN
2169450,US013039666=,2002-02-04,4.130,7.45,13.1507,106.8600,11.201500,NaN,NaN,NaN
2169451,US013039666=,2002-02-05,4.130,7.45,13.1507,107.1600,11.170140,NaN,NaN,NaN
2169452,US013039666=,2002-02-06,4.130,7.45,13.1507,107.1500,11.171183,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1063028,458140AT7=,2025-09-30,5.893,4.90,19.8219,88.4702,11.783915,NaN,NaN,NaN
235338,031162DF4=,2025-09-30,5.663,4.20,26.3918,80.0750,14.463134,NaN,NaN,NaN
1065457,458140AU4=,2025-09-30,4.230,2.60,0.6274,98.9891,0.000000,NaN,NaN,NaN
237097,031162DJ6=,2025-09-30,4.646,4.20,7.4110,97.2248,5.936634,NaN,NaN,NaN


Look at add cusip and other ids to my existing df

In [15]:
yieldbook_indicatives_df = file_dfs["yieldbook_indicatives"]

In [16]:
# daily_df = pd.merge(daily_df, yieldbook_indicatives_df, how="left", on="RIC")
yieldbook_indicatives_smaller_df = yieldbook_indicatives_df[["cusip", "ticker"]]

In [17]:
yield_book_smaller = yield_book[["cusip","MONTH","DtS","Returns", "Hedge_Returns", "Raw_Credit_Excess_Return", "Credit_Excess_Return"]]

In [18]:
daily_with_labels_df = pd.merge(daily_df, yieldbook_indicatives_smaller_df, on="RIC", how="left")

In [19]:
daily_with_labels_df['MONTH'] = daily_with_labels_df['Trade Date'].dt.to_period("M")
daily_with_labels_df

,RIC,Trade Date,Close Yield,CurrentCoupon,YearsToMaturity,Universal Close Price,Effective Duration,P/B,P/E,Market Leverage,cusip,ticker,MONTH
0,001055AY8=,2018-10-31,4.906,4.75,30.1945,97.5490,15.805380,NaN,NaN,NaN,001055AY8,AFL,2018-10
1,001055AY8=,2018-11-01,4.876,4.75,30.1096,98.0120,15.840122,NaN,NaN,NaN,001055AY8,AFL,2018-11
2,001055AY8=,2018-11-02,4.921,4.75,30.1096,97.3170,15.788298,NaN,NaN,NaN,001055AY8,AFL,2018-11
3,001055AY8=,2018-11-05,4.890,4.75,30.1096,97.8090,15.821726,NaN,NaN,NaN,001055AY8,AFL,2018-11
4,001055AY8=,2018-11-06,4.846,4.75,30.1096,98.4920,15.872676,NaN,NaN,NaN,001055AY8,AFL,2018-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2193622,US046686063=,2025-09-24,5.016,5.50,14.2877,104.8913,9.748470,NaN,NaN,NaN,17275RAF9,CSCO,2025-09
2193623,US046686063=,2025-09-25,5.130,5.50,14.2877,103.7125,9.717756,NaN,NaN,NaN,17275RAF9,CSCO,2025-09
2193624,US046686063=,2025-09-26,5.111,5.50,14.2877,103.9054,9.723097,NaN,NaN,NaN,17275RAF9,CSCO,2025-09
2193625,US046686063=,2025-09-29,5.040,5.50,14.2877,104.6399,9.742171,NaN,NaN,NaN,17275RAF9,CSCO,2025-09


Position df and merge with yield_book

In [20]:
daily_with_labels_df = pd.merge(daily_with_labels_df, yield_book_smaller, on=["cusip", "MONTH"], how="left")
daily_with_labels_df = daily_with_labels_df[1:]
daily_with_labels_df = daily_with_labels_df.iloc[:, [0,10,11,1,2,3,4,5,6,7,8,9,12,13,14,15,16,17]]

In [21]:
daily_with_labels_df.columns

Index(['RIC', 'cusip', 'ticker', 'Trade Date', 'Close Yield', 'CurrentCoupon',
       'YearsToMaturity', 'Universal Close Price', 'Effective Duration', 'P/B',
       'P/E', 'Market Leverage', 'MONTH', 'DtS', 'Returns', 'Hedge_Returns',
       'Raw_Credit_Excess_Return', 'Credit_Excess_Return'],
      dtype='object')

Testing grounds for some differences

In [22]:
daily_with_labels_df["Yield Delta"] = daily_with_labels_df.groupby(daily_with_labels_df["MONTH"])["Close Yield"].diff()
daily_with_labels_df["Universal Close Price Delta"] = daily_with_labels_df.groupby(daily_with_labels_df["MONTH"])["Universal Close Price"].diff()
daily_with_labels_df["Effective Duration Delta"] = daily_with_labels_df.groupby(daily_with_labels_df["MONTH"])["Effective Duration"].diff()

Careful to fillna only where it made sense to (first of month)

In [23]:
daily_with_labels_df["Yield Delta"] = daily_with_labels_df["Yield Delta"].fillna(0)
daily_with_labels_df["Universal Close Price Delta"] = daily_with_labels_df["Universal Close Price Delta"].fillna(0)
daily_with_labels_df["Effective Duration Delta"] = daily_with_labels_df["Effective Duration Delta"].fillna(0)

In [24]:
daily_with_labels_df

,RIC,cusip,ticker,Trade Date,Close Yield,CurrentCoupon,YearsToMaturity,Universal Close Price,Effective Duration,P/B,...,Market Leverage,MONTH,DtS,Returns,Hedge_Returns,Raw_Credit_Excess_Return,Credit_Excess_Return,Yield Delta,Universal Close Price Delta,Effective Duration Delta
1,001055AY8=,001055AY8,AFL,2018-11-01,4.876,4.75,30.1096,98.0120,15.840122,NaN,...,NaN,2018-11,2488.191310,0.02576,0.020111,0.005649,0.000002,0.000,0.0000,0.000000
2,001055AY8=,001055AY8,AFL,2018-11-02,4.921,4.75,30.1096,97.3170,15.788298,NaN,...,NaN,2018-11,2488.191310,0.02576,0.020111,0.005649,0.000002,0.045,-0.6950,-0.051823
3,001055AY8=,001055AY8,AFL,2018-11-05,4.890,4.75,30.1096,97.8090,15.821726,NaN,...,NaN,2018-11,2488.191310,0.02576,0.020111,0.005649,0.000002,-0.031,0.4920,0.033428
4,001055AY8=,001055AY8,AFL,2018-11-06,4.846,4.75,30.1096,98.4920,15.872676,NaN,...,NaN,2018-11,2488.191310,0.02576,0.020111,0.005649,0.000002,-0.044,0.6830,0.050950
5,001055AY8=,001055AY8,AFL,2018-11-07,4.852,4.75,30.1096,98.3980,15.865797,NaN,...,NaN,2018-11,2488.191310,0.02576,0.020111,0.005649,0.000002,0.006,-0.0940,-0.006879
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2197376,US046686063=,17275RAF9,CSCO,2025-09-24,5.016,5.50,14.2877,104.8913,9.748470,NaN,...,NaN,2025-09,941.174818,0.01135,0.015386,-0.004036,-0.000004,0.009,-0.0949,-0.002322
2197377,US046686063=,17275RAF9,CSCO,2025-09-25,5.130,5.50,14.2877,103.7125,9.717756,NaN,...,NaN,2025-09,941.174818,0.01135,0.015386,-0.004036,-0.000004,0.114,-1.1788,-0.030713
2197378,US046686063=,17275RAF9,CSCO,2025-09-26,5.111,5.50,14.2877,103.9054,9.723097,NaN,...,NaN,2025-09,941.174818,0.01135,0.015386,-0.004036,-0.000004,-0.019,0.1929,0.005341
2197379,US046686063=,17275RAF9,CSCO,2025-09-29,5.040,5.50,14.2877,104.6399,9.742171,NaN,...,NaN,2025-09,941.174818,0.01135,0.015386,-0.004036,-0.000004,-0.071,0.7345,0.019074


In [25]:
daily_with_labels_df.columns

Index(['RIC', 'cusip', 'ticker', 'Trade Date', 'Close Yield', 'CurrentCoupon',
       'YearsToMaturity', 'Universal Close Price', 'Effective Duration', 'P/B',
       'P/E', 'Market Leverage', 'MONTH', 'DtS', 'Returns', 'Hedge_Returns',
       'Raw_Credit_Excess_Return', 'Credit_Excess_Return', 'Yield Delta',
       'Universal Close Price Delta', 'Effective Duration Delta'],
      dtype='object')

Looking to see any potential outliers in the data that might skew results

In [26]:
columns_to_check = ['Close Yield','CurrentCoupon','Universal Close Price','Effective Duration','P/B','P/E', 'Market Leverage','Raw_Credit_Excess_Return']
daily_with_labels_cleaned_df = daily_with_labels_df.copy()
for i in daily_with_labels_df[columns_to_check]:
    print(daily_with_labels_df.groupby('cusip')[i].max().sort_values(ascending=False).head(10))

cusip
126408GU1    315.9797
110122DK1     92.7140
20030NCU3     37.7200
20030NCS8     37.7200
292505AD6     22.3760
292480AJ9     21.5290
292480AK6     19.8720
292480AM2     16.6210
674599DF9     16.4360
292480AL4     16.2628
Name: Close Yield, dtype: float64
cusip
674599EA9    8.875
84265VAA3    7.500
29273RAJ8    7.500
487836AT5    7.450
406216AY7    7.450
962166BR4    7.375
247361ZZ4    7.375
023551AJ3    7.300
023551AM6    7.125
02005NBR0    7.100
Name: CurrentCoupon, dtype: float64
cusip
92343VBT0    170.7080
87612EAU0    168.7000
494368BC6    164.9633
406216AY7    164.2500
68389XAE5    162.0220
002819AC4    161.7980
002824AV2    161.7770
17275RAD4    159.5140
911312AJ5    159.3320
20030NAC5    159.2287
Name: Universal Close Price, dtype: float64
cusip
91913YAL4    146.093650
30303M8R6     26.202190
78409VAR5     25.744216
20030NDQ1     25.495075
907818FL7     25.391154
037833EA4     25.379255
907818FS2     25.005061
00287YAY5     24.896634
594918CF9     24.448837
458140BK5     24

Group data by cusip and by month. I then sum the daily differences to give the change over the month. This will be used to test OLS later on

In [27]:
required_cols = [
    'Raw_Credit_Excess_Return',
    'Yield Delta',
    'Universal Close Price Delta',
    'Effective Duration Delta'
]
filtered_df = daily_with_labels_df.groupby('cusip').filter(lambda x: x[required_cols].dropna().shape[0] >= 2)
excess_credit_return_df = (filtered_df.groupby(['cusip', 'MONTH'])["Raw_Credit_Excess_Return"].sum()).to_frame()
yield_delta_df = (filtered_df.groupby(['cusip', 'MONTH'])["Yield Delta"].sum()).to_frame()
universal_close_df = (filtered_df.groupby(['cusip', 'MONTH'])["Universal Close Price Delta"].sum()).to_frame()
effective_duration_df = (filtered_df.groupby(['cusip', 'MONTH'])["Effective Duration Delta"].sum()).to_frame()
# type(yield_delta_df), type(universal_close_df), type(effective_duration_df)
model_params_by_cusip = pd.concat([excess_credit_return_df, yield_delta_df, universal_close_df, effective_duration_df], axis=1)
model_params_by_cusip = model_params_by_cusip[model_params_by_cusip.index.get_level_values('MONTH') > "2017-12"]
model_params_by_cusip

Raw_Credit_Excess_Return  Yield Delta  \
cusip     MONTH                                            
001055AY8 2018-11                  0.124281       -0.104   
          2018-12                 -0.365134       -0.182   
          2019-01                  0.613410       -0.208   
          2019-02                  0.281477        0.045   
          2019-03                 -0.206003       -0.283   
...                                     ...          ...   
98978VAV5 2025-05                 -0.357969       -0.772   
          2025-06                 -0.005381       -0.771   
          2025-07                  0.154129       -0.760   
          2025-08                 -0.077166       -0.978   
          2025-09                  0.147444       -0.908   

                   Universal Close Price Delta  Effective Duration Delta  
cusip     MONTH                                                           
001055AY8 2018-11                       1.6410                  0.118962  
          2018-12                       2.9200                  0.215652  
          2019-01                       3.4960                  0.229043  
          2019-02                      -0.7970                 -0.046530  
          2019-03                       4.9640                  0.312672  
...                                        ...                       ...  
98978VAV5 2025-05                      15.0539                 -5.491911  
          2025-06                      14.2191                 -5.539684  
          2025-07                      14.2363                 -5.521757  
          2025-08                      15.7836                 -5.477704  
          2025-09                      14.4925                 -5.532425  

[87757 rows x 4 columns]

create a formula to intake a df and output the regression using these three deltas

In [28]:
rows = []
def fit_ols(df):
    model = smf.ols(
        formula='Raw_Credit_Excess_Return ~ Q("Yield Delta") + Q("Universal Close Price Delta") + Q("Effective Duration Delta")',
        data=df,
        missing='drop'
        ).fit()
    rows.append({
        'Intercept': model.params['Intercept'],
        'Yield Delta': model.params['Q("Yield Delta")'],
        'Price Delta': model.params['Q("Universal Close Price Delta")'],
        'Duration Delta': model.params['Q("Effective Duration Delta")'],
    })

model_params_by_cusip = model_params_by_cusip.apply(fit_ols, axis=1)
rows_df = pd.DataFrame(rows)

In [29]:
rows_df.index = model_params_by_cusip.index
rows_df

Intercept  Yield Delta  Price Delta  Duration Delta
cusip     MONTH                                                       
001055AY8 2018-11   0.033428    -0.003477     0.054856        0.003977
          2018-12  -0.038011     0.006918    -0.110992       -0.008197
          2019-01   0.046060    -0.009580     0.161025        0.010550
          2019-02   0.171695     0.007726    -0.136841       -0.007989
          2019-03  -0.007979     0.002258    -0.039606       -0.002495
...                      ...          ...          ...             ...
98978VAV5 2025-05  -0.001385     0.001070    -0.020856        0.007609
          2025-06  -0.000023     0.000018    -0.000326        0.000127
          2025-07   0.000657    -0.000499     0.009348       -0.003626
          2025-08  -0.000275     0.000268    -0.004333        0.001504
          2025-09   0.000608    -0.000552     0.008813       -0.003364

[87757 rows x 4 columns]

Testing output to see if this is a sensible next step

In [30]:
copy = rows_df.copy()
copy = copy.reset_index()
copy = copy.groupby(['cusip']).mean().sort_index()
copy = copy.drop('MONTH', axis=1)
copy

,Intercept,Yield Delta,Price Delta,Duration Delta
cusip,,,,
001055AY8,-0.000923,-0.000925,0.016922,0.000948
001055BJ0,-0.000192,0.000302,-0.003819,0.002069
001084AS1,-0.006746,-0.008522,-0.033559,-0.017284
00108WAM2,0.000185,0.000149,-0.001677,0.000457
00115AAM1,0.000106,0.000044,0.003271,0.001530
...,...,...,...,...
98956PAV4,0.000094,-0.000485,0.002563,-0.000248
98956PAZ5,-0.000213,-0.000103,-0.002483,-0.000305
98956PBA9,-0.000706,0.000702,0.000282,0.004820


In [31]:
daily_with_labels_cleaned_df

,RIC,cusip,ticker,Trade Date,Close Yield,CurrentCoupon,YearsToMaturity,Universal Close Price,Effective Duration,P/B,...,Market Leverage,MONTH,DtS,Returns,Hedge_Returns,Raw_Credit_Excess_Return,Credit_Excess_Return,Yield Delta,Universal Close Price Delta,Effective Duration Delta
1,001055AY8=,001055AY8,AFL,2018-11-01,4.876,4.75,30.1096,98.0120,15.840122,NaN,...,NaN,2018-11,2488.191310,0.02576,0.020111,0.005649,0.000002,0.000,0.0000,0.000000
2,001055AY8=,001055AY8,AFL,2018-11-02,4.921,4.75,30.1096,97.3170,15.788298,NaN,...,NaN,2018-11,2488.191310,0.02576,0.020111,0.005649,0.000002,0.045,-0.6950,-0.051823
3,001055AY8=,001055AY8,AFL,2018-11-05,4.890,4.75,30.1096,97.8090,15.821726,NaN,...,NaN,2018-11,2488.191310,0.02576,0.020111,0.005649,0.000002,-0.031,0.4920,0.033428
4,001055AY8=,001055AY8,AFL,2018-11-06,4.846,4.75,30.1096,98.4920,15.872676,NaN,...,NaN,2018-11,2488.191310,0.02576,0.020111,0.005649,0.000002,-0.044,0.6830,0.050950
5,001055AY8=,001055AY8,AFL,2018-11-07,4.852,4.75,30.1096,98.3980,15.865797,NaN,...,NaN,2018-11,2488.191310,0.02576,0.020111,0.005649,0.000002,0.006,-0.0940,-0.006879
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2197376,US046686063=,17275RAF9,CSCO,2025-09-24,5.016,5.50,14.2877,104.8913,9.748470,NaN,...,NaN,2025-09,941.174818,0.01135,0.015386,-0.004036,-0.000004,0.009,-0.0949,-0.002322
2197377,US046686063=,17275RAF9,CSCO,2025-09-25,5.130,5.50,14.2877,103.7125,9.717756,NaN,...,NaN,2025-09,941.174818,0.01135,0.015386,-0.004036,-0.000004,0.114,-1.1788,-0.030713
2197378,US046686063=,17275RAF9,CSCO,2025-09-26,5.111,5.50,14.2877,103.9054,9.723097,NaN,...,NaN,2025-09,941.174818,0.01135,0.015386,-0.004036,-0.000004,-0.019,0.1929,0.005341
2197379,US046686063=,17275RAF9,CSCO,2025-09-29,5.040,5.50,14.2877,104.6399,9.742171,NaN,...,NaN,2025-09,941.174818,0.01135,0.015386,-0.004036,-0.000004,-0.071,0.7345,0.019074


In [57]:
row1 = rows_df.copy()

In [ ]:
rows_df = row1.copy()

In [ ]:
# rows_df.columns
# rows_df.index.get_level_values(1)
# print(rows_df.index.get_level_values(1).unique().sort_values())
# ax = rows_df.plot(kind='line', figsize=(12,8))
# plt.show()
rows_df.sort_index(inplace=True)
rows_unstacked_df = rows_df.unstack(level='cusip')
rows_unstacked_df.plot(kind='line')
plt.xlabel('Date')
plt.ylabel('Values')
plt.legend().remove()
plt.figure(figsize=(20,8))
plt.show()